# M4 - Phân tích thủ công 50 mẫu ViHOS

## Mục tiêu

Phân tích thủ công 50 comment từ tập Train để hiểu:

- Nội dung của comment.
- Gold span được đánh dấu ở đâu.
- Comment có bao nhiêu span.
- Vì sao đoạn đó được xem là hate/offensive.
- Những điểm khiến mẫu dễ hoặc khó.
- Các hiện tượng ngôn ngữ thường xuất hiện trong ViHOS.

50 mẫu được chọn cố định:
- 10 comment không có span.
- 18 comment có đúng 1 span.
- 22 comment có từ 2 span trở lên.

In [1]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "DOCS" else current_dir

data_root = project_root / "DATA" / "ViHOS_official"
analysis_root = project_root / "ANALYSIS"

train_path = (
    data_root
    / "train_span_extraction"
    / "train.csv"
)

syllable_path = (
    data_root
    / "train_sequence_labeling"
    / "syllable"
    / "train_BIO_syllable.csv"
)

df_train = pd.read_csv(train_path)
df_syllable = pd.read_csv(syllable_path)

print(len(df_train))
print(df_syllable["sentence_id"].nunique())

8844
8844


In [2]:
span_counts = (
    df_syllable
    .assign(is_span_start=df_syllable["Tag"].eq("B-T"))
    .groupby("sentence_id")["is_span_start"]
    .sum()
)

In [3]:
span_counts.value_counts().sort_index()

is_span_start
0     4552
1     1972
2      992
3      535
4      310
5      191
6       98
7       64
8       47
9       31
10      22
11       6
12      11
13       3
14       2
15       3
18       2
19       1
20       1
23       1
Name: count, dtype: int64

In [5]:
manual_pool = df_train[["content"]].copy()

manual_pool["sentence_id"] = manual_pool.index

manual_pool["span_count"] = (
    manual_pool["sentence_id"]
    .map(span_counts)
    .fillna(0)
    .astype(int)
)

manual_pool.head(10)

,content,sentence_id,span_count
0,Dừa lắm :)),0,0
1,Bấp bênh vl thế,1,1
2,Chắc cũng biết ko tồn tại đc bao lâu nữa nên c...,2,1
3,Thấy chán ad page này kiến thức thì nông cản c...,3,6
4,Giang Giang Đỗ Thị Ngọc Hà trend mới kìa kìa,4,0
5,đcm 😒 sau có con cho hút cỏ chữa bệnh chứ đéo ...,5,5
6,Má nứng quá aiu ơi,6,2
7,Chụp cái ảnh thôi làm đéo gì đâu mà phiền,7,1
8,Tăng theo giờ 😭😭😭,8,0
9,Cực kì thuyết phục :c,9,0


In [6]:
clean_pool = manual_pool[manual_pool['span_count']==0]
single_pool = manual_pool[manual_pool['span_count']==1]
multi_pool = manual_pool[manual_pool['span_count']>=2]

print(len(clean_pool))
print(len(single_pool))
print(len(multi_pool))

4552
1972
2320


In [27]:
clean_pool = clean_pool.sample(
    frac=1,
    random_state=221
).reset_index(drop=True)

single_pool = single_pool.sample(
    frac=1,
    random_state=222
).reset_index(drop=True)

multi_pool = multi_pool.sample(
    frac=1,
    random_state=223
).reset_index(drop=True)

In [29]:
for pool in [clean_pool, single_pool, multi_pool]:
    pool["SELECT"] = ""
    pool["NOTE"] = ""

columns = [
    "sentence_id",
    "content",
    "span_count",
    "SELECT",
    "NOTE"
]

clean_pool = clean_pool[columns]
single_pool = single_pool[columns]
multi_pool = multi_pool[columns]

In [30]:
output_dir = project_root / "DATA"

clean_path = output_dir / "M4_clean_candidates.xlsx"
single_path = output_dir / "M4_single_span_candidates.xlsx"
multi_path = output_dir / "M4_multi_span_candidates.xlsx"

clean_pool.to_excel(
    clean_path,
    index=False
)

single_pool.to_excel(
    single_path,
    index=False
)

multi_pool.to_excel(
    multi_path,
    index=False
)

print("Đã tạo:")
print(clean_path)
print(single_path)
print(multi_path)

Đã tạo:
c:\cs221-FinalTerm-ViHOS\DATA\M4_clean_candidates.xlsx
c:\cs221-FinalTerm-ViHOS\DATA\M4_single_span_candidates.xlsx
c:\cs221-FinalTerm-ViHOS\DATA\M4_multi_span_candidates.xlsx


In [31]:
from openpyxl import load_workbook

excel_files = [
    clean_path,
    single_path,
    multi_path
]

for file_path in excel_files:

    wb = load_workbook(file_path)
    ws = wb.active

    # Cố định hàng tiêu đề
    ws.freeze_panes = "A2"

    # Độ rộng cột
    ws.column_dimensions["A"].width = 15
    ws.column_dimensions["B"].width = 100
    ws.column_dimensions["C"].width = 12
    ws.column_dimensions["D"].width = 12
    ws.column_dimensions["E"].width = 35

    # Cho content xuống dòng trong ô
    for row in ws.iter_rows(min_row=2):
        row[1].alignment = row[1].alignment.copy(
            wrap_text=True,
            vertical="top"
        )

    wb.save(file_path)

print("Đã format 3 file Excel.")

C:\Users\Nguyen\AppData\Local\Temp\ipykernel_2912\1040630013.py:26: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  row[1].alignment = row[1].alignment.copy(


Đã format 3 file Excel.


In [32]:
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\cs221-FinalTerm-ViHOS")

data_dir = project_root / "DATA"
analysis_dir = project_root / "ANALYSIS"

analysis_dir.mkdir(exist_ok=True)

In [33]:
clean = pd.read_excel(data_dir / "M4_clean_candidates_SELECTED.xlsx")

single = pd.read_excel(data_dir / "M4_single_span_candidates_SELECTED.xlsx")

multi = pd.read_excel(data_dir / "M4_multi_span_candidates_SELECTED.xlsx")

In [34]:
def get_selected(df):
    mask = (
        df["SELECT"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("X")
    )

    return df[mask].copy()

In [36]:
clean_selected = get_selected(clean)
single_selected = get_selected(single)

multi = pd.read_excel(
    data_dir / "M4_multi_span_candidates_SELECTED.xlsx",
    header=5
)

multi_selected = multi[
    ["sentence_id", "content", "span_count"]
].copy()

In [37]:
print("Clean:", len(clean_selected))
print("Single:", len(single_selected))
print("Multi:", len(multi_selected))

Clean: 10
Single: 18
Multi: 22


In [3]:
from pathlib import Path
import pandas as pd

# =========================
# 1. ĐƯỜNG DẪN
# =========================
project_root = Path(r"C:\cs221-FinalTerm-ViHOS")
data_dir = project_root / "DATA"
analysis_dir = project_root / "ANALYSIS"
analysis_dir.mkdir(exist_ok=True)

# =========================
# 2. ĐỌC 3 FILE ĐÃ CHỐT
# =========================
clean = pd.read_excel(
    data_dir / "M4_clean_candidates_SELECTED.xlsx"
)

single = pd.read_excel(
    data_dir / "M4_single_span_candidates_SELECTED.xlsx"
)

# File multi có header bắt đầu ở hàng Excel thứ 6
multi = pd.read_excel(
    data_dir / "M4_multi_span_candidates_SELECTED.xlsx",
    header=5
)

# =========================
# 3. LẤY CÁC CÂU ĐƯỢC CHỌN
# =========================
def get_selected(df):
    mask = (
        df["SELECT"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("X")
    )
    return df.loc[
        mask,
        ["sentence_id", "content", "span_count"]
    ].copy()

clean_selected = get_selected(clean)
single_selected = get_selected(single)

# File multi đã chỉ chứa đúng 22 câu được chọn
multi_selected = multi[
    ["sentence_id", "content", "span_count"]
].copy()

# =========================
# 4. KIỂM TRA QUOTA
# =========================
assert len(clean_selected) == 10, f"Clean phải là 10, hiện có {len(clean_selected)}"
assert len(single_selected) == 18, f"Single phải là 18, hiện có {len(single_selected)}"
assert len(multi_selected) == 22, f"Multi phải là 22, hiện có {len(multi_selected)}"

# =========================
# 5. GẮN NHÓM
# =========================
clean_selected["group"] = "clean"
single_selected["group"] = "single"
multi_selected["group"] = "multi"

# =========================
# 6. GHÉP THÀNH 50 CÂU
# =========================
manual_50 = pd.concat(
    [clean_selected, single_selected, multi_selected],
    ignore_index=True
)

# Xáo cố định
manual_50 = (
    manual_50
    .sample(frac=1, random_state=221)
    .reset_index(drop=True)
)

# Tạo mã S01 -> S50
manual_50.insert(
    0,
    "sample_id",
    [f"S{i:02d}" for i in range(1, 51)]
)

# Kiểm tra cuối
assert len(manual_50) == 50
assert manual_50["sentence_id"].nunique() == 50

# =========================
# 7. XUẤT FILE EXCEL CUỐI
# =========================
output_path = analysis_dir / "manual_50_selected.xlsx"

manual_50.to_excel(
    output_path,
    index=False
)

print("PASS")
print("Clean :", (manual_50["group"] == "clean").sum())
print("Single:", (manual_50["group"] == "single").sum())
print("Multi :", (manual_50["group"] == "multi").sum())
print("Total :", len(manual_50))
print("\nĐã tạo:", output_path)

PASS
Clean : 10
Single: 18
Multi : 22
Total : 50

Đã tạo: C:\cs221-FinalTerm-ViHOS\ANALYSIS\manual_50_selected.xlsx


In [4]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill

# =========================================================
# 1. ĐƯỜNG DẪN
# =========================================================
project_root = Path(r"C:\cs221-FinalTerm-ViHOS")

analysis_dir = project_root / "ANALYSIS"

manual_path = analysis_dir / "manual_50_selected.xlsx"

bio_path = (
    project_root
    / "DATA"
    / "ViHOS_official"
    / "train_sequence_labeling"
    / "syllable"
    / "train_BIO_syllable.csv"
)

output_path = analysis_dir / "manual_50_analysis.xlsx"


# =========================================================
# 2. ĐỌC 50 MẪU + BIO TRAIN
# =========================================================
manual_50 = pd.read_excel(manual_path)
df_syllable = pd.read_csv(bio_path)


# =========================================================
# 3. HÀM LẤY GOLD SPAN TỪ BIO
# =========================================================
def get_gold_spans(sentence_id):

    rows = df_syllable[
        df_syllable["sentence_id"] == sentence_id
    ][["Word", "Tag"]]

    spans = []
    current_span = []

    for word, tag in zip(rows["Word"], rows["Tag"]):

        if tag == "B-T":

            if current_span:
                spans.append(" ".join(current_span))

            current_span = [str(word)]

        elif tag == "I-T":

            if current_span:
                current_span.append(str(word))

        else:  # O

            if current_span:
                spans.append(" ".join(current_span))
                current_span = []

    if current_span:
        spans.append(" ".join(current_span))

    return " | ".join(spans)


# =========================================================
# 4. TẠO GOLD SPANS
# =========================================================
manual_50["gold_spans"] = manual_50["sentence_id"].apply(
    get_gold_spans
)


# =========================================================
# 5. THÊM CỘT PHÂN TÍCH THỦ CÔNG
# =========================================================
manual_50["difficulty"] = ""
manual_50["phenomenon"] = ""
manual_50["manual_comment"] = ""

# Chia 25 câu / thành viên
manual_50["member"] = [
    "Member 1" if i < 25 else "Member 2"
    for i in range(len(manual_50))
]


# =========================================================
# 6. SẮP XẾP CỘT
# =========================================================
manual_50 = manual_50[
    [
        "sample_id",
        "sentence_id",
        "content",
        "group",
        "span_count",
        "gold_spans",
        "difficulty",
        "phenomenon",
        "manual_comment",
        "member"
    ]
]


# =========================================================
# 7. KIỂM TRA
# =========================================================
assert len(manual_50) == 50
assert manual_50["sentence_id"].nunique() == 50

# Clean phải không có gold span
assert (
    manual_50.loc[
        manual_50["group"] == "clean",
        "gold_spans"
    ] == ""
).all()

print("Số mẫu:", len(manual_50))
print()
print(manual_50["group"].value_counts())


# =========================================================
# 8. XUẤT EXCEL
# =========================================================
manual_50.to_excel(
    output_path,
    index=False
)


# =========================================================
# 9. FORMAT CHO DỄ PHÂN TÍCH
# =========================================================
wb = load_workbook(output_path)
ws = wb.active

ws.title = "Manual Analysis"
ws.freeze_panes = "A2"

# Độ rộng cột
widths = {
    "A": 11,   # sample_id
    "B": 13,   # sentence_id
    "C": 80,   # content
    "D": 12,   # group
    "E": 12,   # span_count
    "F": 40,   # gold_spans
    "G": 15,   # difficulty
    "H": 30,   # phenomenon
    "I": 70,   # manual_comment
    "J": 15    # member
}

for col, width in widths.items():
    ws.column_dimensions[col].width = width


# Header
for cell in ws[1]:
    cell.font = Font(bold=True)
    cell.fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7"
    )
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True
    )


# Wrap text toàn bộ
for row in ws.iter_rows(min_row=2):

    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True
        )

    ws.row_dimensions[row[0].row].height = 55


wb.save(output_path)

print("\nPASS M4.2")
print("Đã tạo:", output_path)

Số mẫu: 50

group
multi     22
single    18
clean     10
Name: count, dtype: int64

PASS M4.2
Đã tạo: C:\cs221-FinalTerm-ViHOS\ANALYSIS\manual_50_analysis.xlsx
